# 06 - XGBoost Classifier

Train an XGBoost classifier to predict `performance_category`, then evaluate it on the same stratified 80/20 split used by the other model notebooks. The preprocessing pipeline is included in the saved artifact so the backend can accept the original feature columns. Outcome-derived fields are excluded to reduce target leakage.

This notebook also records accuracy, macro precision/recall, and macro-F1 in `models/model_metrics.json`. Macro-F1 gives each performance category equal weight, which is useful when the classes are imbalanced.

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from backend.model_utils import TargetLabelClassifier

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'student_performance_processed.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
target = 'performance_category'
excluded = {'student_id', target, 'risk_level', 'final_marks', 'passed'}
features = [column for column in df.columns if column not in excluded]
X = df[features].copy()
y = df[target].astype(str)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Data shape: {df.shape}')
print(f'Feature count: {len(features)}')
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')
print('Class counts in training data:')
print(y_train.value_counts().sort_index())

Data shape: (5000, 20)
Feature count: 15
Train rows: 4000 | Test rows: 1000
Class counts in training data:
performance_category
average      1665
excellent     571
good         1478
poor          286
Name: count, dtype: int64


In [3]:
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_features),
])

xgb_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=2,
    verbosity=0,
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', TargetLabelClassifier(xgb_classifier)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('XGBoost training complete.')

XGBoost training complete.


In [4]:
metrics = {
    'model': 'XGBoost',
    'accuracy': round(accuracy_score(y_test, predictions), 4),
    'precision_macro': round(precision_score(y_test, predictions, average='macro', zero_division=0), 4),
    'recall_macro': round(recall_score(y_test, predictions, average='macro', zero_division=0), 4),
    'f1_macro': round(f1_score(y_test, predictions, average='macro', zero_division=0), 4),
}
print(metrics)
print('\nClassification report:')
print(classification_report(y_test, predictions, zero_division=0))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, predictions))

transformed_names = model.named_steps['preprocessor'].get_feature_names_out()
importance_values = model.named_steps['classifier'].feature_importances_
feature_importance = pd.DataFrame({
    'feature': transformed_names,
    'importance': importance_values,
}).sort_values('importance', ascending=False)
print('\nTop 15 XGBoost feature importances:')
display(feature_importance.head(15))

{'model': 'XGBoost', 'accuracy': 0.63, 'precision_macro': 0.6194, 'recall_macro': 0.5503, 'f1_macro': 0.5756}

Classification report:
              precision    recall  f1-score   support

     average       0.66      0.73      0.69       417
   excellent       0.70      0.50      0.58       143
        good       0.59      0.63      0.61       369
        poor       0.53      0.35      0.42        71

    accuracy                           0.63      1000
   macro avg       0.62      0.55      0.58      1000
weighted avg       0.63      0.63      0.63      1000


Confusion matrix:
[[303   2  90  22]
 [  2  71  70   0]
 [109  29 231   0]
 [ 46   0   0  25]]

Top 15 XGBoost feature importances:


,feature,importance
7,numeric__quiz_average,0.152855
1,numeric__previous_marks,0.104514
8,numeric__assignment_average,0.080604
3,numeric__study_hours_per_week,0.051717
4,numeric__study_days_per_week,0.050000
17,categorical__study_frequency_low,0.046701
2,numeric__attendance_pct,0.045050
9,numeric__missed_assignments,0.044254
16,categorical__study_frequency_high,0.041764
5,numeric__study_consistency_pct,0.037902


In [5]:
model_path = MODEL_DIR / 'student_performance_xgboost.pkl'
with model_path.open('wb') as model_file:
    pickle.dump(model, model_file)

metrics_path = MODEL_DIR / 'model_metrics.json'
saved_metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else []
saved_metrics = [row for row in saved_metrics if row.get('model') != metrics['model']]
saved_metrics.append(metrics)
metrics_path.write_text(json.dumps(saved_metrics, indent=2))

print(f'Saved XGBoost pipeline: {model_path}')
print(f'Updated model comparison: {metrics_path}')

Saved XGBoost pipeline: /workspaces/LearnIQ---Personalized-Learning-Performance-Prediction-System/models/student_performance_xgboost.pkl
Updated model comparison: /workspaces/LearnIQ---Personalized-Learning-Performance-Prediction-System/models/model_metrics.json


## Conclusion and Metric Guide

`n_estimators=300` sets 300 boosting rounds, not 300 epochs. In this four-class multiclass setup, XGBoost builds class-specific trees during each round. `max_depth=4` limits tree depth; `learning_rate=0.05` controls each round's contribution. `subsample` and `colsample_bytree` sample training rows and columns to help control overfitting.

Accuracy is the share of test rows classified correctly. Macro precision, recall, and F1 calculate each category's score separately and average categories equally. The current XGBoost results are 0.6300 accuracy and 0.5756 macro-F1 on one stratified 80/20 holdout split; this is a baseline, not a guarantee of future performance. The API's displayed confidence is the model's estimated class probability and has not been separately calibrated.

The `predict_proba` values are estimated probabilities for the performance categories; the highest value determines the predicted category. Feature importance summarizes how the trees used transformed features for splits and does not show causal effects. The API selects the available model with the highest recorded macro-F1 after restart; XGBoost is saved as `models/student_performance_xgboost.pkl` and its comparison row appears in the frontend.